In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import numpy as np
import pandas as pd
import torch

In [3]:
from IPython.display import HTML

HTML('''
<center style="font-family: Calibri; font-weight:bold; letter-spacing: 0px; color:#2e4053; border-radius:5px; font-size:380%; text-align:center; padding:3.0px; background: #7DCEA0; border-bottom: 3px solid #2c3e50; border-top: 8px solid #2c3e50 white-space: pre-wrap; word-wrap: break-word;">
  Backpropagation 👀 Coding from Scratch
</center>

<center style="font-family: Calibri; font-weight:bold; letter-spacing: 0px; color:#2e4053; border-radius:5px; font-size:140%; text-align:center; padding:3.0px; background: #7DCEA0; border-bottom: 6px solid #2c3e50; border-top: 8px solid #2c3e50;">
  Playground Series - Season 4, Episode 11
</center>

<center>
  <img src="https://i0.wp.com/analyticsarora.com/wp-content/uploads/2021/09/Understand-The-Backpropagation-Algorithm-Interview-Question.png?resize=800%2C600&ssl=1" width="800"/>
</center>
''')


# <p style= "font-family: Calibri; font-weight:bold; letter-spacing: 0px; color:#2e4053; border-radius:5px; font-size:120%; text-align:left;padding:3.0px; background: #7DCEA0 ; border-bottom: 4px solid #2c3e50; border-top: 4px solid k" > 1. Get and preview the datasets </p> 

In [4]:
df = pd.DataFrame([[8,8,4],[7,9,5],[6,10,6],[5,12,7]], columns=['cgpa', 'profile_score', 'lpa'])

In [5]:
df

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,7


In [6]:
# Input Features
X = df[['cgpa', 'profile_score']].values
# Target
y = df[['lpa']].values
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

In [7]:
def initialize_parameters(layer_dims):
  
  np.random.seed(3)
  parameters = {}
  L = len(layer_dims)         

  for l in range(1, L):

    parameters['W' + str(l)] = torch.ones((layer_dims[l-1], layer_dims[l]))*0.1
    parameters['b' + str(l)] = torch.zeros((layer_dims[l], 1))
      

  return parameters

In [8]:
initialize_parameters([2,2,1])

{'W1': tensor([[0.1000, 0.1000],
         [0.1000, 0.1000]]),
 'b1': tensor([[0.],
         [0.]]),
 'W2': tensor([[0.1000],
         [0.1000]]),
 'b2': tensor([[0.]])}

## <p style= "font-family: Calibri; font-weight:bold; letter-spacing: 0px; color:#2e4053; border-radius:5px; font-size:120%; text-align:left;padding:3.0px; background: #7DCEA0 ; border-bottom: 4px solid #2c3e50; border-top: 4px solid k" > 1.1 Initializing all weigths=0.1 and bias=0 </p> 


**Parameter Initialization in Neural Networks: Step-by-Step Example**

This summary illustrates how variables are allocated when using a simple neural network parameter initialization function for a model with architecture `[2, 2, 1]`.

| Step | Layer(s)       | Variable | Shape   | Allocation Example     | Content                    |
|-------|----------------|----------|---------|-----------------------|----------------------------|
| 1     | Input → Hidden | `W1`     | (2, 2)  | `np.ones((2,2))*0.1`  | `[[0.1, 0.1], [0.1, 0.1]]`  |
|       |                | `b1`     | (2, 1)  | `np.zeros((2,1))`     | `[[0.], [0.]]`             |
| 2     | Hidden → Output| `W2`     | (2, 1)  | `np.ones((2,1))*0.1`  | `[[0.1], [0.1]]`           |
|       |                | `b2`     | (1, 1)  | `np.zeros((1,1))`     | `[[0.]]`                   |

---

**Key Details:**

- `layer_dims = [2, 2, 1]` means:  
  - Input layer: 2 units  
  - Hidden layer: 2 units  
  - Output layer: 1 unit
| NumPy       | PyTorch             |
| ----------- | ------------------- |
| np.array()  | torch.tensor()      |
| np.dot()    | torch.matmul()      |
| reshape()   | reshape() / view()  |
| dtype=float | dtype=torch.float32 |

- The loop allocates a weight matrix (`W`) and bias vector (`b`) for each layer.

- Weights are set to `0.1`, and biases to `0`, as a simple initialization.

- The final `parameters` dictionary after initialization:  
  - `'W1'`: shape (2, 2)  
  - `'b1'`: shape (2, 1)  
  - `'W2'`: shape (2, 1)  
  - `'b2'`: shape (1, 1)

---

**Summary Table**

| Variable | Shape   | Description                    | Example Value             |
|----------|---------|--------------------------------|---------------------------|
| `W1`     | (2, 2)  | Weights from input to hidden   | `[[0.1, 0.1], [0.1, 0.1]]` |
| `b1`     | (2, 1)  | Bias for hidden layer          | `[[0.], [0.]]`             |
| `W2`     | (2, 1)  | Weights from hidden to output  | `[[0.1], [0.1]]`           |
| `b2`     | (1, 1)  | Bias for output layer          | `[[0.]]`                   |

---

**Practical Tip:**  
Parameter initialization is critical — it controls early model outputs and influences learning stability.  
For reproducibility, the random seed can be set: `np.random.seed(3)`.

---



## <p style= "font-family: Calibri; font-weight:bold; letter-spacing: 0px; color:#2e4053; border-radius:5px; font-size:120%; text-align:left;padding:3.0px; background: #7DCEA0 ; border-bottom: 4px solid #2c3e50; border-top: 4px solid k" > 1.2 Forward propogation from scratch.</p> 

In [9]:
def linear_forward(A_prev, W, b):
  
  Z = torch.matmul(W.T, A_prev) + b
  
  return Z

In [10]:
# Forward Prop
def L_layer_forward(X, parameters):

  A = X
  L = len(parameters) // 2                  # number of layers in the neural network
  
  for l in range(1, L+1):
    A_prev = A 
    Wl = parameters['W' + str(l)]
    bl = parameters['b' + str(l)]
    print("A"+str(l-1)+": ", A_prev)
    print("W"+str(l)+": ", Wl)
    print("b"+str(l)+": ", bl)
    print("--"*20)

    A = linear_forward(A_prev, Wl, bl)
    print("A"+str(l)+": ", A)
    print("**"*20)
          
  return A,A_prev

## <p style= "font-family: Calibri; font-weight:bold; letter-spacing: 0px; color:#2e4053; border-radius:5px; font-size:120%; text-align:left;padding:3.0px; background: #7DCEA0 ; border-bottom: 4px solid #2c3e50; border-top: 4px solid k" > 1.3 Passing first row of dataframe </p>

In [11]:
X_tensor

tensor([[ 8.,  8.],
        [ 7.,  9.],
        [ 6., 10.],
        [ 5., 12.]])

In [12]:
y_tensor

tensor([[4.],
        [5.],
        [6.],
        [7.]])

In [13]:
X = df[['cgpa', 'profile_score']].values[0].reshape(2,1)
X

array([[8],
       [8]])

In [14]:
y = df[['lpa']].values[0][0]
y

4

In [15]:
X_tensor1 = X_tensor[0].reshape(2,1)
X_tensor1

tensor([[8.],
        [8.]])

In [16]:
y_tensor[0][0]

tensor(4.)

## <p style= "font-family: Calibri; font-weight:bold; letter-spacing: 0px; color:#2e4053; border-radius:5px; font-size:120%; text-align:left;padding:3.0px; background: #7DCEA0 ; border-bottom: 4px solid #2c3e50; border-top: 4px solid k" > 2. Prediction from the first rowm of dataset </p>

In [17]:
X = df[['cgpa', 'profile_score']].values[0].reshape(2,1) # Shape(no of features, no. of training example)
y = df[['lpa']].values[0][0]

# Parameter initialization
parameters = initialize_parameters([2,2,1])

y_hat,A1 = L_layer_forward(X_tensor1, parameters)

A0:  tensor([[8.],
        [8.]])
W1:  tensor([[0.1000, 0.1000],
        [0.1000, 0.1000]])
b1:  tensor([[0.],
        [0.]])
----------------------------------------
A1:  tensor([[1.6000],
        [1.6000]])
****************************************
A1:  tensor([[1.6000],
        [1.6000]])
W2:  tensor([[0.1000],
        [0.1000]])
b2:  tensor([[0.]])
----------------------------------------
A2:  tensor([[0.3200]])
****************************************


In [18]:
y_hat = y_hat
y_hat

tensor([[0.3200]])

## <p style= "font-family: Calibri; font-weight:bold; letter-spacing: 0px; color:#2e4053; border-radius:5px; font-size:120%; text-align:left;padding:3.0px; background: #7DCEA0 ; border-bottom: 4px solid #2c3e50; border-top: 4px solid k" > 3. Updation of parameters.</p>

In [19]:
def update_parameters(parameters,y,y_hat,A1,X):
  parameters['W2'][0][0] = parameters['W2'][0][0] + (0.001 * 2 * (y - y_hat)*A1[0][0])
  parameters['W2'][1][0] = parameters['W2'][1][0] + (0.001 * 2 * (y - y_hat)*A1[1][0])
  parameters['b2'][0][0] = parameters['W2'][1][0] + (0.001 * 2 * (y - y_hat))

  parameters['W1'][0][0] = parameters['W1'][0][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][0][0]*X[0][0])
  parameters['W1'][0][1] = parameters['W1'][0][1] + (0.001 * 2 * (y - y_hat)*parameters['W2'][0][0]*X[1][0])
  parameters['b1'][0][0] = parameters['b1'][0][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][0][0])

  parameters['W1'][1][0] = parameters['W1'][1][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][1][0]*X[0][0])
  parameters['W1'][1][1] = parameters['W1'][1][1] + (0.001 * 2 * (y - y_hat)*parameters['W2'][1][0]*X[1][0])
  parameters['b1'][1][0] = parameters['b1'][1][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][1][0])

In [20]:
update_parameters(parameters,y_tensor[0][0],y_hat,A1,X_tensor)

In [21]:
parameters

{'W1': tensor([[0.1066, 0.1058],
         [0.1066, 0.1058]]),
 'b1': tensor([[0.0008],
         [0.0008]]),
 'W2': tensor([[0.1118],
         [0.1118]]),
 'b2': tensor([[0.1191]])}

## <p style= "font-family: Calibri; font-weight:bold; letter-spacing: 0px; color:#2e4053; border-radius:5px; font-size:120%; text-align:left;padding:3.0px; background: #7DCEA0 ; border-bottom: 4px solid #2c3e50; border-top: 4px solid k" > 4. passing 2nd row the parameters.</p>

In [22]:
X_tensor

tensor([[ 8.,  8.],
        [ 7.,  9.],
        [ 6., 10.],
        [ 5., 12.]])

In [23]:
X_tensor2 = X_tensor[1].reshape(2,1)


# Parameter initialization
parameters = initialize_parameters([2,2,1])

y_hat,A1 = L_layer_forward(X_tensor2,parameters)
y_hat = y_hat[0][0]

update_parameters(parameters,y_tensor[1][0],y_hat,A1,X)

parameters

A0:  tensor([[7.],
        [9.]])
W1:  tensor([[0.1000, 0.1000],
        [0.1000, 0.1000]])
b1:  tensor([[0.],
        [0.]])
----------------------------------------
A1:  tensor([[1.6000],
        [1.6000]])
****************************************
A1:  tensor([[1.6000],
        [1.6000]])
W2:  tensor([[0.1000],
        [0.1000]])
b2:  tensor([[0.]])
----------------------------------------
A2:  tensor([[0.3200]])
****************************************


{'W1': tensor([[0.1086, 0.1086],
         [0.1086, 0.1086]]),
 'b1': tensor([[0.0011],
         [0.0011]]),
 'W2': tensor([[0.1150],
         [0.1150]]),
 'b2': tensor([[0.1243]])}

In [24]:
X_tensor

tensor([[ 8.,  8.],
        [ 7.,  9.],
        [ 6., 10.],
        [ 5., 12.]])

In [25]:
y_tensor

tensor([[4.],
        [5.],
        [6.],
        [7.]])

In [26]:
# epochs implementation

parameters = initialize_parameters([2,2,1])
epochs = 5

for i in range(epochs):

  Loss = []

  for j in range(df.shape[0]):

    X = X_tensor[j] # Shape(no of features, no. of training example)
    y = y_tensor[j][0]

    # Parameter initialization


    y_hat,A1 = L_layer_forward(X,parameters)
    y_hat = y_hat[0][0]

    update_parameters(parameters,y,y_hat,A1,X_tensor)

    Loss.append((y-y_hat)**2)

  print('Epoch - ',i+1,'Loss - ',np.array(Loss).mean())

parameters

A0:  tensor([8., 8.])
W1:  tensor([[0.1000, 0.1000],
        [0.1000, 0.1000]])
b1:  tensor([[0.],
        [0.]])
----------------------------------------
A1:  tensor([[1.6000, 1.6000],
        [1.6000, 1.6000]])
****************************************
A1:  tensor([[1.6000, 1.6000],
        [1.6000, 1.6000]])
W2:  tensor([[0.1000],
        [0.1000]])
b2:  tensor([[0.]])
----------------------------------------
A2:  tensor([[0.3200, 0.3200]])
****************************************
A0:  tensor([7., 9.])
W1:  tensor([[0.1066, 0.1058],
        [0.1066, 0.1058]])
b1:  tensor([[0.0008],
        [0.0008]])
----------------------------------------
A1:  tensor([[1.7061, 1.6930],
        [1.7061, 1.6930]])
****************************************
A1:  tensor([[1.7061, 1.6930],
        [1.7061, 1.6930]])
W2:  tensor([[0.1118],
        [0.1118]])
b2:  tensor([[0.1191]])
----------------------------------------
A2:  tensor([[0.5005, 0.4976]])
****************************************
A0:  tensor(

{'W1': tensor([[0.3347, 0.3054],
         [0.3347, 0.3054]]),
 'b1': tensor([[0.0293],
         [0.0293]]),
 'W2': tensor([[0.4527],
         [0.4527]]),
 'b2': tensor([[0.4563]])}